# Chapter 31: Structures: pin-jointed trusses

The code for this chapter from [*Matrices in Practice*](https://trgoodale.github.io/matrix-book/part-6/31-trusses.html), by Tom Goodale.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trgoodale/matrix-book/blob/main/code/python/31-trusses.ipynb)

Run the cells in order, from the top: later cells use variables set in earlier ones. Change the numbers and run them again to see what happens. The code needs NumPy, SciPy and Matplotlib (and pandas in a few chapters), all of which Colab already has.

Code licensed under the [MIT licence](https://trgoodale.github.io/matrix-book/LICENSE.md).

## In code

The function below builds the equilibrium matrix from lists of joints, members and supports, following the model above, and solves it. For the Pratt bridge it prints the force in each member with the 20 kN load at the middle of the span.

In [ ]:
import numpy as np

def truss(P, members, supports, loads):
    """Equilibrium matrix A and right-hand side for a pin-jointed truss.
    Unknowns: member forces (tension positive), then reactions."""
    P = np.array(P, dtype=float)
    reactions = [(j, d) for j, kind in supports.items()
                 for d in ([0, 1] if kind == "pin" else [1])]
    A = np.zeros((2 * len(P), len(members) + len(reactions)))
    for k, (i, j) in enumerate(members):
        e = (P[j] - P[i]) / np.linalg.norm(P[j] - P[i])
        A[2 * i:2 * i + 2, k] += e        # member pulls joint i towards j
        A[2 * j:2 * j + 2, k] -= e        # and joint j towards i
    for r, (j, d) in enumerate(reactions):
        A[2 * j + d, len(members) + r] = 1
    b = np.zeros(2 * len(P))
    for j, (fx, fy) in loads.items():
        b[2 * j:2 * j + 2] = [-fx, -fy]
    return A, b

# Example 1
A, b = truss([[0, 0], [4, 0], [2, 2]], [(0, 1), (0, 2), (1, 2)],
             {0: "pin", 1: "roller"}, {2: (0, -10)})
print(np.linalg.solve(A, b))

# Pratt bridge, 20 kN at the middle of the bottom chord
joints = [[0, 0], [4, 0], [8, 0], [12, 0], [16, 0], [4, 4], [8, 4], [12, 4]]
members = [(0, 1), (1, 2), (2, 3), (3, 4), (5, 6), (6, 7), (1, 5), (2, 6),
           (3, 7), (0, 5), (4, 7), (5, 2), (7, 2)]
A, b = truss(joints, members, {0: "pin", 4: "roller"}, {2: (0, -20)})
print("rank", np.linalg.matrix_rank(A), "of", A.shape)
f = np.linalg.solve(A, b)
for (i, j), N in zip(members, f):
    print(f"member {i}-{j}: {N:7.2f} kN")

## Exercises

The code exercises from the [end of the chapter](https://trgoodale.github.io/matrix-book/part-6/31-trusses.html#sec-trusses-exercises). Worked solutions are in the notebook `solutions-part-6.ipynb`, and in the [book's solutions](https://trgoodale.github.io/matrix-book/appendices/solutions-6.html#sec-solutions-trusses).

### Exercise 7

For the Pratt bridge, move a 20 kN load to each of the three interior joints of the bottom chord in turn. For each member, find the largest tension and the largest compression it ever carries. Which members must be designed for both?

In [ ]:
# Your code here